# Auditer la conformite visuelle — ce que le smoke test ne voit pas

*Notebook compagnon du Parcours 1 (Copilot Gutenberg) — axe conformite visuelle.*
*A completer du grain sur la derive de chaine* (`mesurer-la-derive-dun-copilot.ipynb`) *: celui-ci mesurait la perte d'information dans une chaine de transformations ; celui-la mesure la conformite du rendu final.*

## La question

Un Copilot Gutenberg genere ou modifie des templates de pages. Un AI Forms
affiche un formulaire sur le frontend. La verification la plus courante est un
**smoke test structurel** : la page repond (simule ici par un statut 200),
la balise `<main>` est presente et non vide, un element d'action (`<a>` ou
`<button>`) est present. Ce test **passe sur une page dont le rendu visuel est
casse** :

- un CTA colore en bleu `#007bff` (primaire Bootstrap) au lieu du ton charte ;
- un texte sur fond trop clair, sous le seuil de contraste WCAG ;
- un CTA semi-transparent, sans classe de bouton, qui paraît desactive.

Ce notebook convertit cette constatation en **mesure reproductible**. Il
construit quatre pages synthetiques portant des violations deliberees, ecrit
les detecteurs dedies (contraste WCAG, dominance des primaires, affordance des
CTA), et montre que le smoke test est **aveugle** aux trois classes de defauts.

> **Lien avec** `docs/reference/verification-verte-systeme-casse.md`. Ce
> document de reference decrit le motif « la sonde ment » pour la classe
> *systeme* (HTTP 200 sur un site casse). Le present notebook en demontre la
> classe *visuelle* : meme structure, classe differente, detecteurs
> differents. Le motif recidive, et c'est la lecon.

### Comment lire ce notebook

Le notebook suit une pente unique : partir d'une charte ecrite comme des
donnees, construire quatre pages dont trois portent un defaut delibere,
montrer qu'une sonde usuelle ne voit aucun des trois defauts, puis batir
trois detecteurs — un par classe de defaut — et les croiser dans une
matrice finale. Chaque etape produit une sortie committee, et chaque
chiffre cite dans les sections d'interpretation figure dans l'une de ces
sorties : rien n'est affirme qui ne soit imprime par une cellule.

Trois habitudes de lecture avant d'avancer. D'abord, les detecteurs
proposes sont des **schemas pedagogiques**, pas des outils de production :
ils lisent le HTML source par expressions regulieres, la ou un audit reel
interrogerait le rendu calcule d'un navigateur. Ce choix assume un but :
rendre visible la mecanique de chaque detection, ligne par ligne, sans
cache technique. Ensuite, la question du notebook n'est pas « ces pages
sont-elles conformes ? » mais « **quelle sonde voit quel defaut ?** » —
la conformite des quatre pages est connue d'avance, c'est la variable
fixee; la sonde est la variable libre. Enfin, le fil rouge est un
incident reel, anonymise et documente ailleurs dans le depot : des
audits consecutifs d'un site ont declare « operationnel » sur la foi
d'une sonde structurelle, alors que le rendu etait casse. La matrice
finale est la reponse methodologique a cet incident — une sonde par
classe de defaut, jamais une sonde unique pour toutes.
**Prerequis et perimetre technique**, pour situer l'effort : tout le
notebook tourne sur la bibliotheque standard (`re` seul, en verite) —
aucune dependance, aucun reseau, aucune cle, aucun rendu. Ce n'est pas
une economie de moyen, c'est un choix de lisibilite : un detecteur de
quatorze lignes que l'etudiant peut reecrire de memoire vaut mieux,
pour apprendre, qu'une brique d'accessibilite industrielle qui rend
son verdict en boite noire. La transposition a un projet reel ne
demande ni plus ni moins que de remplir le dictionnaire `CHARTE` avec
les vraies valeurs de la vraie charte — le jour ou cette charte
existe sous forme de donnees. C'est la vraie difficulte de l'exercice,
et elle n'est pas technique : la plupart des chartes visuelles
n'existent qu'en prose, et le premier travail d'audit est de les
rendre executables.

## 1. La charte est une specification, pas un sentiment

Pour qu'un detecteur de conformite puisse dire « conforme » ou « non
conforme », il faut une **charte explicitement declarée** : des couleurs, des
polices, des regles d'usage. On travaille sur une instance synthetique — la
fiction editoriale « Maison Valmont » — dont la charte est donnee ci-dessous.
**Aucune couleur ici n'est empruntee a une identite reelle** ; la charte est
l'instance minimale qui rend la notion de conformite definissable.

In [1]:
# La charte de Maison Valmont (synthetique) : un dictionnaire de contrats.
CHARTE = {
    "fond":           "#faf6ef",  # creme
    "texte":          "#2d4a3e",  # vert profond
    "accent":         "#c9a96e",  # dore
    "cta_fond":       "#2d4a3e",  # CTA principal : fond vert profond
    "cta_texte":      "#ffffff",  # CTA principal : texte blanc
    "police_titres":  "Playfair Display",
}

# Les couleurs primaires Bootstrap : le marqueur d'une esthetique SaaS
# generique qui s'importe avec le framework, pas avec la charte.
PRIMAIRES_BOOTSTRAP = {
    "#007bff",  # blue (primary)
    "#dc3545",  # danger (red)
    "#28a745",  # success (green)
    "#ffc107",  # warning (yellow)
    "#17a2b8",  # info (cyan)
    "#6c757d",  # secondary (gray)
}

print("Charte Maison Valmont definie : " + str(len(CHARTE)) + " contrats couleur.")
print("Marqueur primaire Bootstrap : " + str(len(PRIMAIRES_BOOTSTRAP)) + " couleurs guettees.")

Charte Maison Valmont definie : 6 contrats couleur.
Marqueur primaire Bootstrap : 6 couleurs guettees.


### Lire la charte comme un contrat executable

La sortie annonce la symetrie : **6 contrats couleur** cote charte,
**6 couleurs guettees** cote marqueur Bootstrap. Deux listes de six,
mais de natures opposees — et cette opposition est le coeur de la
section. Le dictionnaire `CHARTE` est une **allowlist partielle** : il
nonce ce qui doit exister (le creme de fond, le vert profond du texte,
le dore en accent). `PRIMAIRES_BOOTSTRAP` est une **denylist** : elle
nonce ce qui ne doit jamais apparaitre. Un audit de charte complet
croise les deux — ce qui manque (une couleur de charte absente des
pages) et ce qui depasse (une couleur etrangere presente).

Pourquoi la denylist cible-t-elle precisement les six primaires d'un
framework CSS, plutot que « toute couleur hors charte » ? Parce que le
signal recherche n'est pas la simple non-conformite : c'est le
**temoin d'import**. Ces six teintes ne se choisissent pas une par une —
elles arrivent en bloc quand un agent branche un framework et laisse
ses styles par defaut parler a sa place. Une couleur unique hors charte
peut etre une decision (ou une erreur isolee) ; un `#007bff` en CTA
accompagne de `#dc3545` et `#28a745` en badges est une **absence de
decision**. La denylist est donc un marqueur de provenance autant
qu'une regle de palette — et cette double lecture guidera le detecteur
de la section 5.

Derniere consequence pratique de « charte = donnees » : la charte devient
**diffusable et verifiable**. Un document qui dit « tons feutres, esprit
litteraire » ne se teste pas ; un dictionnaire de six couples cle-valeur
se teste. Toute charte qui ne peut pas se reduire a cette forme ne sera
jamais auditee par machine — seulement appreciee par un humain, ce qui
est exactement la situation que le notebook veut eviter.
Noter enfin ce que ces six cles **n'encodent pas** : aucune hierarchie
de tailles, aucun rythme d'espacement, aucune regle de composition, et
une seule entree typographique (la police des titres) — pas de graisses,
pas d'echelles. Un audit machine ne couvrira jamais que ce que la
charte encode : une charte reduite aux couleurs produit un audit
reduit aux couleurs. La circularite est vertueuse, pas limitante —
pour qu'une intention visuelle soit auditee, il faut d'abord qu'elle
soit ecrite assez precisement pour etre fausse. Le travail preparatoire
d'un audit de conformite est donc toujours le meme : convertir la
prose de charte en donnees testables, et assumer que ce qui ne se
convertit pas restera, faute de sonde, au jugement humain — en le
nommant comme tel dans le rapport, plutot qu'en le dissimulant derriere
un pourcentage global flatteur.

La charte n'est pas negociale : toute couleur hors palette est un suspect.
Le detecteur de dominance (section 5) se resume a croiser les couleurs
declaredes dans la page avec l'ensemble `PRIMAIRES_BOOTSTRAP` — rien de plus.

## 2. Quatre pages, un seul defaut chacune

On construit quatre fragments HTML. Trois portent **une violation visuelle
deliberee, et une seule** — pour isoler chaque classe de defaut. La quatrieme
est la reference conforme. Toutes ont une balise `<main>` non vide et un
element d'action : le smoke test les declarera saines.

In [2]:
PAGE_CONFORME = '''<html><body>
<main>
  <h1 style="font-family:'Playfair Display'; color:#2d4a3e">La Collection Valmont</h1>
  <p style="color:#2d4a3e">Trois recits pour la rentree litteraire de septembre.</p>
  <a class="btn cta-principal" href="/collection" style="background-color:#2d4a3e; color:#ffffff">Decouvrir la collection</a>
</main>
</body></html>'''

PAGE_PRIMAIRES = '''<html><body>
<main>
  <h1 style="color:#2d4a3e">La Collection Valmont</h1>
  <p style="color:#2d4a3e">Trois recits pour la rentree litteraire de septembre.</p>
  <a class="btn" href="/collection" style="background-color:#007bff; color:#ffffff">Decouvrir</a>
  <span class="badge" style="background-color:#dc3545; color:#ffffff">Nouveau</span>
  <span class="badge" style="background-color:#28a745; color:#ffffff">En stock</span>
</main>
</body></html>'''

PAGE_CONTRASTE = '''<html><body>
<main>
  <h1 style="color:#2d4a3e">La Collection Valmont</h1>
  <p style="color:#c9a96e">Lisez notre presentation detaillee de la rentree litteraire.</p>
  <a class="btn" href="/collection" style="background-color:#2d4a3e; color:#ffffff">Decouvrir</a>
</main>
</body></html>'''

PAGE_AFFORDANCE = '''<html><body>
<main>
  <h1 style="color:#2d4a3e">La Collection Valmont</h1>
  <p style="color:#2d4a3e">Trois recits pour la rentree litteraire de septembre.</p>
  <a href="/collection" style="color:#2d4a3e; opacity:0.5">Decouvrir la collection</a>
  <a href="/newsletter" style="color:rgba(45,74,62,0.4)">S'abonner a la lettre</a>
</main>
</body></html>'''

PAGES = {
    "conforme":  PAGE_CONFORME,
    "primaires": PAGE_PRIMAIRES,
    "contraste": PAGE_CONTRASTE,
    "affordance": PAGE_AFFORDANCE,
}
print(str(len(PAGES)) + " pages synthetiques preparees (3 defaillantes, 1 conforme).")

4 pages synthetiques preparees (3 defaillantes, 1 conforme).


### Quatre pages, une seule variable chacune

La construction des pages est un protocole experimental deguise en
fixture. Les quatre pages partagent le meme titre, le meme paragraphe,
la meme action « Decouvrir » : chaque page defaillante ne differe de la
page conforme que par **le seul attribut qui porte son defaut**. La
page `contraste` change une couleur (le vert du paragraphe devient
dore) ; la page `primaires` ajoute un CTA bleu et deux badges colores ;
la page `affordance` retire la classe de bouton et pose deux
transparences. Rien d'autre ne bouge.

Pourquoi cette discipline d'isolement ? Parce que la matrice finale ne
sera lisible que si chaque case rouge a **une seule cause possible**.
Si une page cumulait trois defauts, on verrait bien qu'elle echoue —
mais on ne saurait pas quel detecteur attrape quel defaut. L'isolement
transforme la matrice en experience : quand `contraste` echoue au
detecteur de contraste et a aucun autre, l'echec est attribuable au
defaut seul, pas a une contamination voisine. C'est le meme geste que
celui d'un banc de test qui varie une entree a la fois.

Noter aussi ce que la page conforme enseigne par comparaison directe :
le dore `#c9a96e` est **dans la charte** (cle `accent`), et pourtant la
page `contraste` est defaillante. Le defaut de contraste n'est donc pas
un defaut de palette — c'est un defaut d'**usage** : une couleur
legitime employee pour un role interdit (texte courant sur fond creme).
Cette distinction — couleur illegitime contre usage illegitime —
structurera deux detecteurs differents, et la section suivante la
rend explicite.
Deux choix de support meritent d'etre explicites. Pourquoi des pages
**synthetiques** plutot que des captures d'un vrai site ? Pour la
reproductibilite : ces quatre pages vivent dans le notebook, quiconque
rejoue les cellules retrouve exactement ces sources, et chaque defaut
est pose par construction — donc connaissable independamment de tout
rendu. Une capture reelle ne se rejoue pas ; un defaut reel n'est
connu que par le diagnostic qu'on en a fait, et ce diagnostic peut
etre faux. Et pourquoi des pages si **minimalistes** ? Parce que dans
une vraie page — cinquante elements, trois feuilles de style, un
framework — le defaut serait noye ; ici, chaque attribut des sources
est lisible a l'oeil et la matrice finale reste verifiable a la main.
La miniature est le prix de la demonstration ; l'echelle reelle est
l'affaire de la transposition, pas du schema.

- **`primaires`** : CTA en bleu Bootstrap, badges en rouge et vert Bootstrap. Couleurs hors charte.
- **`contraste`** : paragraphe en dore `#c9a96e` sur creme `#faf6ef` — or sur creme, peu lisible.
- **`affordance`** : deux CTA en `<a>` brut, sans `.btn`, l'un a `opacity:0.5`, l'autre en `rgba(...,0.4)`.

### Pourquoi ces trois defauts et pas d'autres

Les trois defauts portes par les pages ne sont pas trois variations
esthetiques prises au hasard : chacun represente une **classe d'echec
observee** quand une interface est generee ou reconfiguree par un agent.
La classe `primaires` : l'agent importe un framework et laisse ses
couleurs par defaut s'exprimer — defaut de **provenance**. La classe
`contraste` : l'agent choisit une couleur de la charte mais la place
dans un role ou elle est illisible — defaut d'**usage**. La classe
`affordance` : l'agent produit un lien fonctionnel au sens du DOM, mais
visuellement muet — defaut de **signal**.

Cette taxonomie a une consequence directe sur l'architecture de
l'audit : chaque classe exige son propre detecteur, parce qu'aucun
indicateur unique ne couvre les trois. Un detecteur de palette ne voit
pas le dore sur creme (le dore est conforme). Un detecteur de contraste
ne voit pas le CTA semi-transparent (le contraste du texte reste
correct). Un detecteur d'affordance ne voit ni l'un ni l'autre. La
matrice de la section 7 est la mise en forme de cette impossibilite —
trois colonnes de detection parce que trois classes independantes.

Retenir aussi le choix du mot « affordance » plutot que « style des
boutons » : l'affordance est la propriete qui dit a l'utilisateur
**qu'une action est possible ici**. Un lien texte `opacity:0.5` n'a
rien d'invalide au sens du DOM ni du contraste — il a un defaut de
invitation. C'est la classe la plus subjective des trois, et c'est
pourquoi son detecteur (section 6) sera aussi le plus conventionnel :
deux canaux mesurables y remplacent le jugement visuel.
L'ordre de la section est aussi une observation de frequence. Quand
une interface est generee ou reconfiguree par un agent, la classe
`primaires` apparait en premier — c'est le defaut de l'installation
qui branche un framework et n'ecrase pas ses defauts. La classe
`affordance` vient ensuite — l'agent produit du DOM correct et oublie
que l'utilisateur ne lit pas le DOM. La classe `contraste` vient en
dernier, et c'est la plus sournoise des trois : la couleur fautive
est **dans la charte**, aucun detecteur de palette ne l'attrapera
jamais, et l'oeil la pardonne la moitie du temps. Une taxonomie
d'erreurs qui ne serait pas ordonnee par cette escalade de
discretion — du defaut voyant au defaut invisible — en oublierait la
question operationnelle : par quoi commencer, et surtout par quoi
finir, quand les derniers defauts sont ceux qu'aucune sonde evidente
ne signale.

## 3. Le smoke test : la sonde qui ne voit rien

Le smoke test est la verification minimale qu'on deploye apres une modification
de template. Il ne regarde que la **structure** : la balise `<main>` est-elle
presente et non vide ? Un element d'action est-il present ? La page est-elle
« servie » (simule par un statut 200) ?

In [3]:
import re

def smoke_test(html_str):
    """Verifie la presence structurelle. Ne regarde AUCUNE couleur."""
    main_non_vide = bool(re.search(r"<main[^>]*>\s*\S", html_str, re.IGNORECASE))
    action_present = bool(re.search(r"<(a|button)[^>]*>", html_str, re.IGNORECASE))
    return {
        "statut":      200,
        "structure":   "PASS" if main_non_vide else "FAIL",
        "action":      "PASS" if action_present else "FAIL",
    }

for nom, page in PAGES.items():
    print(nom.ljust(11), smoke_test(page))

conforme    {'statut': 200, 'structure': 'PASS', 'action': 'PASS'}
primaires   {'statut': 200, 'structure': 'PASS', 'action': 'PASS'}
contraste   {'statut': 200, 'structure': 'PASS', 'action': 'PASS'}
affordance  {'statut': 200, 'structure': 'PASS', 'action': 'PASS'}


### Le smoke test repond a une autre question

Relire la sortie lentement : quatre lignes, quatre fois le meme
dictionnaire — `statut: 200`, `structure: PASS`, `action: PASS` — y
compris sur les trois pages defaillantes. Aucune discrimination. La
tentation est de conclure « le smoke test est casse » ; la lecture
exacte est plus instructive : **le smoke test fonctionne parfaitement,
et il ne mesure rien de ce qui est en jeu ici**. Sa regex cherche un
`<main>` non vide et une balise `<a>` ou `<button>` ; les trois defauts
vivent dans des attributs `style` que la regex ne lit jamais. La sonde
n'est pas en panne — elle repond correctement a une question dont la
reponse etait connue d'avance (« la page repond et possede une
structure »), et on lui fait dire un verdict (« conforme ») qu'elle
n'a jamais promis.

C'est la distinction la plus rentable de tout le notebook : entre une
sonde **fausse** et une sonde **aveugle**. Une sonde fausse mesure mal
ce qu'elle pretend mesurer ; une sonde aveugle mesure bien autre chose.
La sonde aveugle est la plus dangereuse des deux en production, parce
que ses PASS sont tous sinceres : chaque vert est une reponse exacte a
sa question, et l'erreur est uniquement dans la question qu'on lui fait
porter en la citant comme preuve de conformite.

D'ou la regle de lecture qui suit pour toutes les sections : avant
d'invoquer une sonde comme preuve, **nommer la question exacte a
laquelle elle repond**. Le smoke test repond a « la page est-elle
servie et structuree ? » — jamais a « la page est-elle conforme ? ».
Les trois detecteurs qui suivent repondront chacun a une question
etroite, et c'est precisement parce qu'elle est etroite que leurs
verdicts seront utilisables.
Il faut defendre la sonde avant de la demissionner : le smoke test a
une legitimite reelle sur SA question. Il detecte les pannes graves —
page vide, erreur 500, timeout, route absente — et il les detecte
vite, ce qui fait de lui la premiere ligne de tout pipeline de
deploiement. L'erreur de categorie n'invalide pas la sonde ; elle
invalide sa **citation** hors categorie. D'ou la regle operationnelle
qui se degage de cette section, et qui vaut pour tout rapport
d'audit : chaque verdict cite ses sondes par leur nom, et chaque
sonde est definie par la question a laquelle elle repond — non par
l'outil qui l'implemente. Un rapport qui ecrit « conforme (smoke
test) » n'a pas fait une mesure ; il a confondu un nom d'outil avec
une question, et la confusion se propage a tous ceux qui le liront
sans rejouer.

**Le smoke test passe sur les quatre pages**, y compris les trois cassees
visuellement. C'est exactement le scenario recurrent : un agent regenere un
template, le smoke test reste vert, le deploiement est declare « operationnel »,
et le rendu public est casse. Le smoke test mesure le **contenant** (la
structure est la), pas la **conformite** (le contenu respecte-t-il la charte ?).

Il nous faut donc trois autres detecteurs, un par classe de defaut visuel.

## 4. Detecteur de contraste (WCAG)

Le contraste percu depend de la **luminance** relative des deux couleurs,
pas de leur teinte. Le ratio WCAG est defini par le W3C : on linearise chaque
canal RGB (correction gamma), on combine en luminance, puis on forme le ratio
`(L_clair + 0,05) / (L_fonce + 0,05)`. Le seuil **AA** est **4,5:1** pour du
texte normal (3:1 pour du grand texte).

In [4]:
def hex_vers_rgb(c):
    c = c.lstrip("#")
    return tuple(int(c[i:i+2], 16) for i in (0, 2, 4))

def _canal_lineaire(c):
    c = c / 255.0
    return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4

def luminance(rgb):
    r, g, b = rgb
    return 0.2126 * _canal_lineaire(r) + 0.7152 * _canal_lineaire(g) + 0.0722 * _canal_lineaire(b)

def contraste_w3c(avant, arriere):
    la = luminance(hex_vers_rgb(avant))
    lr = luminance(hex_vers_rgb(arriere))
    clair, fonce = max(la, lr), min(la, lr)
    return round((clair + 0.05) / (fonce + 0.05), 2)

SEUIL_AA_NORMAL = 4.5

# Quelques mesures de bon sens avant l'audit.
print("Blanc / vert profond (#2d4a3e) : ", contraste_w3c("#ffffff", "#2d4a3e"))
print("Blanc / bleu primaire (#007bff) : ", contraste_w3c("#ffffff", "#007bff"))
print("Dore / creme (#c9a96e sur #faf6ef) : ", contraste_w3c("#c9a96e", "#faf6ef"))

Blanc / vert profond (#2d4a3e) :  9.72
Blanc / bleu primaire (#007bff) :  3.98
Dore / creme (#c9a96e sur #faf6ef) :  2.08


### Pourquoi l'oeil pardonne ce que la formule refuse

Les trois mesures de bon sens meritent plus qu'un coup d'oeil. Blanc
sur vert profond : **9.72** — confortable. Dore sur creme : **2.08** —
sous le seuil AA de **4.5** par plus de moitie. Et la troisieme mesure,
la plus interessante : blanc sur bleu primaire, **3.98** — egalement
sous le seuil. Trois lignes, deux lecons distinctes.

La premiere lecon est physique : pourquoi le dore sur creme, qui parait
« un peu clair » a l'oeil, tombe-t-il a 2.08 ? Parce que la luminance
W3C ne compare pas les valeurs RGB mais leurs **linearisations gamma** :
chaque canal eleve a la puissance 2.4 (via la formule `((c+0.055)/1.055)
** 2.4`, seuil `0.03928` pour les valeurs tres sombres). Cette
transformation ecrase les hautes lumieres : le creme clair et le dore
sont tous deux dans la zone ou la courbe s'aplatit, donc leur ecart
de luminance lineaire est bien plus petit que ne le suggere l'ecart de
teinte. L'oeil, lui, adapte son gain local et rehausse le texte sur
fond clair — il « voit » un contraste que la physique du peripherique
de rendu ne delivrera pas a l'ecran. La formule decrit l'ecran, pas la
sensation : c'est pourquoi le seuil legal tranche la ou l'impression
visuelle reste vague.

La seconde lecon est un defaut cache du banc lui-meme : le bleu
`#007bff`, present dans la page `primaires`, echoue lui aussi le
contraste (3.98 < 4.5). Pourtant la matrice finale dira `primaires` :
contraste PASS. La mesure existe dans cette sortie ; l'audit de la
section 4 ne la verra pas, pour une raison de perimetre qui deviendra
visible a la section suivante. Garder cette tension en tete : elle est
l'illustration integree de la these du notebook — meme bien construit,
un detecteur ne certifie que son perimetre exact.
Precision utile pour relire les seuils : la regle AA distingue le
texte courant (**4.5**) du texte large (**3.0** — plus de 18 points, ou
14 en gras), et le niveau AAA exige **7.0** — valeurs absentes des
sorties parce que le banc n'audite que du texte courant, mais
necessaires des qu'on transpose. La formule elle-meme merite un
arret : c'est un **ratio de luminances lineaires**, sans unite, et le
**+0.05** du numerateur comme du denominateur n'est pas un arrondi —
c'est la garde qui empeche une division par zero quand l'une des
deux couleurs est le noir pur (luminance exactement nulle). Enfin, le
cas **3.98** illustre la zone grise la plus inconfortable : entre 3.98
et 4.5, l'ecart est invisible a l'oeil — aucun humain ne trie ces
deux pages — et c'est precisement pour cela que l'arbitrage se fait
par la regle ecrite et non par l'impression : la ou l'oeil ne
discrimine plus, seule la formule tranche, et elle tranche pour
l'ecran reel, pas pour la sensation.

Le dore sur creme (2,08) est bien sous le seuil — c'est le defaut de la page
`contraste`. Maintenant l'audit lui-meme : on extrait la couleur de chaque
paragraphe et titre, et on la compare au fond declare dans la charte.

In [5]:
def audit_contraste_texte(html_str, fond):
    """Cherche les couleurs des balises de texte, les compare au fond.
    Renvoie le ratio minimal et son verdict (le pire cas decide)."""
    ratios = []
    for m in re.finditer(r'<(p|h[1-6])[^>]*style="([^"]*)"[^>]*>', html_str, re.IGNORECASE):
        style = m.group(2)
        cm = re.search(r"color:\s*(#[0-9a-fA-F]{6})", style)
        if cm:
            ratios.append(contraste_w3c(cm.group(1), fond))
    if not ratios:
        return None, "N/A"
    mini = min(ratios)
    return mini, "PASS" if mini >= SEUIL_AA_NORMAL else "FAIL"

for nom, page in PAGES.items():
    mini, verdict = audit_contraste_texte(page, CHARTE["fond"])
    detail = ("ratio min " + str(mini)) if mini is not None else "aucun texte couleur"
    print(nom.ljust(11), "contraste :", verdict, "(" + detail + ")")

conforme    contraste : PASS (ratio min 9.02)
primaires   contraste : PASS (ratio min 9.02)
contraste   contraste : FAIL (ratio min 2.08)
affordance  contraste : PASS (ratio min 9.02)


### Le pire cas decide, et le perimetre aussi

Deux choix de conception se lisent dans la sortie, l'un visible,
l'autre dans ce qu'elle ne dit pas.

Le choix visible : le verdict porte sur le **minimum** des ratios, pas
sur leur moyenne. La page conforme affiche 9.02 — c'est le vert du
paragraphe sur creme ; la moyenne des textes d'une page serait plus
flatteuse et strictement moins utile : un seul texte illisible rend la
page inutilisable pour qui precise ce texte. `min()` encode cette
asymetrie — le pire cas decide, exactement comme la cellule de la
section 3 l'a deja etabli. Noter au passage d'ou vient ce 9.02 : la
mesure de bon sens de la section precedente donnait 9.72 pour blanc
sur vert — c'est le CTA, balise `<a>`, que ce detecteur ne lit pas.
Il n'audite que les balises `p` et `h1` a `h6` : 9.02 est le couple
texte-courant-sur-fond, 9.72 le couple bouton. Deux questions proches,
deux nombres differents — et seul le premier compte pour ce verdict.

Le choix invisible, c'est le perimetre meme : la regex ne voit que les
styles **inline** sur `p`/`h*`. Les deux badges de la page `primaires`
— blanc sur rouge `#dc3545`, blanc sur vert `#28a745` — ne sont pas
audites : ce sont des `<span>`. Le defaut de contraste reel du bleu
Bootstrap (mesure a 3.98 la cellule precedente) vit sur un `<a>`,
egalement hors regex. La sortie dit `primaires : PASS (ratio min
9.02)` et elle est **sincere dans son perimetre** — le bleu du CTA et
les badges sont ailleurs. Enfin, le fond est passe en argument unique :
tout texte pose sur une tuile, une image ou un fond differant du creme
charte echappe aussi. Un detecteur honnete est un detecteur qui rend
son perimetre explicite — celui-ci le fait par sa signature de
fonction, pas par sa docstring.
Un detail du code porte une decision subtile : quand aucun texte
colore n'est trouve, la fonction renvoie `("N/A", ...)` — **pas**
PASS. La distinction est programmatique et pas rhetorique : une page
sans texte auditable ne peut pas etre certifiee conforme pour son
contraste, elle est **non evaluee** — l'abstention n'est pas une
reussite. Confondre N/A et PASS serait fabriquer des conformites par
silence : la page la plus vide deviendrait la plus conforme. C'est le
genre de detail qui distingue un verdict exploitable d'un score : un
verdict a trois etats (PASS, FAIL, N/A) et chaque etat nomme une
realite differente ; un score a deux etats rapproche ce qu'il faut
distinguer. Le detecteur de la section suivante fera le meme choix —
zero primaire trouvee vaut PASS, mais zero CTA reconnu dans une page
qui en affiche visuellement reste un angle mort, et la sortie l'a
deja revele a qui la relit lentement.

Seule la page `contraste` echoue — son paragraphe dore sur creme. Les trois
autres pages ont un texte lisible. **Ce detecteur voit ce que le smoke test
ignorait.**

## 5. Detecteur de dominance des primaires

Deuxieme classe de defaut : une page qui affiche des couleurs SaaS generiques
(les primaires Bootstrap) la ou la charte exige des tons tamises. Le detecteur
recense les couleurs declarees dans la page et signale celles qui figurent dans
l'ensemble `PRIMAIRES_BOOTSTRAP`.

In [6]:
_HEX = re.compile(r"#([0-9a-fA-F]{6})\b")
_RGB = re.compile(r"rgb\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)")

def couleurs_declarees(html_str):
    trouv = []
    for m in _HEX.finditer(html_str):
        trouv.append("#" + m.group(1).lower())
    for m in _RGB.finditer(html_str):
        trouv.append("#%02x%02x%02x" % (int(m.group(1)), int(m.group(2)), int(m.group(3))))
    return trouv

def audit_dominance(html_str):
    utilisees = couleurs_declarees(html_str)
    primaires = sorted(set(c for c in utilisees if c in PRIMAIRES_BOOTSTRAP))
    return primaires

for nom, page in PAGES.items():
    primaires = audit_dominance(page)
    verdict = "PASS" if not primaires else "FAIL"
    detail = str(len(primaires)) + " primaire(s)" if primaires else "0 primaire"
    print(nom.ljust(11), "dominance :", verdict, "(" + detail + ")")

conforme    dominance : PASS (0 primaire)
primaires   dominance : FAIL (3 primaire(s))
contraste   dominance : PASS (0 primaire)
affordance  dominance : PASS (0 primaire)


### Allowlist, denylist, et la normalisation des notations

La sortie est la plus tranchee du notebook : trois primaires sur la
page `primaires`, zero partout ailleurs. Trois decisions de conception
meritent d'etre lues dans le code avant de croire le verdict.

Premiere decision : le detecteur compare les couleurs **declarees** a
la denylist de la section 1, pas a la charte. Une couleur hors charte
et hors denylist — un violet maison, un gris importe d'ailleurs —
passerait. Le choix est assume : le detecteur traque le temoin
d'import framework, pas l'exhaustivite de conformite. Deuxieme
decision : la normalisation. `#2d4a3e` et `rgb(45, 74, 62)` decrivent
la meme couleur ; sans la conversion rgb-vers-hex, la page
`affordance` — dont le second CTA est pose en `rgba(45,74,62,0.4)` —
compterait ses couleurs dans deux mondes differents. Le `_HEX` porte
un `` apres les six chiffres : sans lui, un identifiant plus long
pourrait livrer six hexadecimaux parasites. Details petits, mais
chacun est la difference entre un detecteur et une intuition.

Troisieme decision, la plus discutable et donc la plus instructive :
`sorted(set(...))` dedouble et ordonne. Le compte « 3 primaire(s) »
signifie **trois teintes distinctes**, pas trois occurrences — un CTA
et dix badges bleus compteraient une seule primaire. Pour un temoin
d'import, c'est le bon choix (une teinte suffit a prouver la
provenance) ; pour mesurer l'**envahissement**, ce serait le mauvais
(il faudrait compter les occurrences). Le detecteur repond a « cette
page porte-t-elle l'esthetique importee ? », pas a « a quel point ? ».
Deux questions legitimes, deux compteurs differents — le verdict texte
n'en porte qu'un.
Que faudrait-il pour un veritable detecteur d'exhaustivite — chaque
couleur declaree presente dans la charte ? L'idee se heurte vite a la
realite des interfaces fonctionnelles : les gris neutres (bordures,
separateurs, textes secondaires), les etats (hover, focus, disabled),
les ombres — aucun projet reel ne vit avec six couleurs exactement.
L'allowlist stricte est donc inapplicable telle quelle ; ce qui
s'applique est un **graphe de tolerance** (charte, plus une zone
neutre bornee en luminance), dont l'ecriture est un vrai travail de
design, pas un raffinement technique. C'est la raison profonde du
choix denylist de ce banc : il ne faut pas une charte enrichie pour
tourner, et il attrape le signal le plus grave — l'esthetique
importee — des le premier jour. L'audit complet des deux cotes
(denylist pour la provenance, allowlist tolerante pour la
completude) est une progression, pas un tout-or-rien.

Seule la page `primaires` echoue, avec ses trois couleurs Bootstrap. Notons
que ces couleurs sont **parfaitement lisibles** (blanc sur bleu Bootstrap a un
contraste correct pour un grand bouton) : le detecteur de contraste ne les
voit pas comme un defaut. C'est volontaire — **chaque detecteur capture une
classe distincte**, et aucun ne peut seul certifier la conformite globale.

## 6. Detecteur d'affordance des CTA

Troisieme classe : un lien qui devrait se comporter comme un bouton mais n'en
a ni la classe (`.btn`) ni l'opacite pleine — il semble desactive. Le detecteur
repere les elements d'action (liens et boutons au texte oriente action) et
verifie qu'ils portent une classe de bouton et une opacite suffisante.

In [7]:
TEXTES_ACTION = (
    "decouvrir", "lire la suite", "soumettre", "s'abonner",
    "en savoir plus", "telecharger", "je m'inscris",
)

def extraire_cta(html_str):
    cta = []
    for m in re.finditer(r"<(a|button)([^>]*)>(.*?)</\1>", html_str, re.DOTALL | re.IGNORECASE):
        tag, attrs, texte = m.group(1), m.group(2).lower(), re.sub(r"<[^>]+>", "", m.group(3)).strip()
        texte_bas = texte.lower()
        if ("btn" in attrs) or ("cta" in attrs) or any(t in texte_bas for t in TEXTES_ACTION):
            cta.append({"tag": tag, "attrs": attrs, "texte": texte})
    return cta

def _opacite_effective(attrs):
    om = re.search(r"opacity:\s*([0-9.]+)", attrs)
    if om:
        return float(om.group(1))
    rm = re.search(r"rgba\([^)]*,\s*([0-9.]+)\s*\)", attrs)
    if rm:
        return float(rm.group(1))
    return 1.0

def audit_affordance(html_str):
    verdicts = []
    for cta in extraire_cta(html_str):
        a_bouton = ("btn" in cta["attrs"]) or (cta["tag"] == "button")
        opac = _opacite_effective(cta["attrs"])
        ok = a_bouton and opac >= 0.7
        verdicts.append((cta["texte"][:30], a_bouton, opac, "PASS" if ok else "FAIL"))
    return verdicts

for nom, page in PAGES.items():
    verdicts = audit_affordance(page)
    global_v = "PASS" if all(v[3] == "PASS" for v in verdicts) and verdicts else "FAIL"
    print(nom.ljust(11), "affordance :", global_v, "(" + str(len(verdicts)) + " CTA)")

conforme    affordance : PASS (1 CTA)
primaires   affordance : PASS (1 CTA)
contraste   affordance : PASS (1 CTA)
affordance  affordance : FAIL (2 CTA)


### Deux canaux pour un signal d'action

Le detecteur d'affordance est le plus conventionnel des trois, et
c'est le seul qui doit l'etre : il remplace un jugement visuel (« on
voit que c'est cliquable ») par deux canaux mesurables. Le premier
canal est **semantique** : la presence de `btn` ou `cta` dans les
attributs, ou un texte de la liste `TEXTES_ACTION`. Le second est
**physique** : une opacite effective au-dessus du seuil 0.7. Le
verdict exige les deux — un CTA reconnu mais fantomatique echoue, un
lien opaque mais sans marqueur d'action echoue aussi. La page
`affordance` echoue deux fois, une fois par CTA, par deux syntaxes du
meme vice : `opacity:0.5` et l'alpha d'un `rgba(...,0.4)`.

Le detail qui compte est `_opacite_effective` : elle lit d'abord
`opacity:`, puis, a defaut, l'alpha du `rgba()`. Deux notations, un
phenomene — la semi-transparence — et un detecteur qui aurait oublie
la seconde notation raterait exactement la moitie du defaut de la
page. C'est l'anti-pattern docstring du banc : une docstring aurait
pu dire « verifie l'opacite », le code, lui, enumere ses deux sources.
Une sonde se juge sur ce qu'elle enumere, pas sur ce qu'elle decrit.

Deux frontieres assumer : la liste `TEXTES_ACTION` est francaise et
fermee — un CTA au libelle hors liste, sans classe, echappe a la
reconnaissance (il aura alors zero verdict, pas un verdict FAIL : la
sortie affiche « 0 CTA » si rien n'est reconnu, et la page passe — un
angle mort a connaitre). Et le seuil **0.7** est arbitraire dans sa
valeur exacte mais monotone dans son ordre : 0.4 puis 0.5 puis 0.7
puis 1.0 — ce qui discrimine la page est l'ordre, pas la constante.
Un audit reel calibrerait ce seuil sur des captures reelles ; le
schema, lui, n'a besoin que de la monotonic.
Le croisement des deux canaux n'est pas un luxe de rigueur : il
protege des faux verdicts des deux bords. Un CTA avec la classe
`.btn` mais rendu par un framework dont les styles ne se chargent
pas — classe presente, affordance absente — est attrape par le canal
physique. Un lien parfaitement opaque et lisible mais pose sans
marqueur — texte hors liste, sans classe — est attrape par le canal
semantique... a la condition que son libelle soit dans la liste, et
c'est la breche deja signalee. En pratique, le canal semantique est
le plus fiable des deux parce que les conventions de classes (`.btn`,
`.cta`, `.button`) sont des **contrats d'equipe** : si l'equipe ecrit
sa convention, le detecteur la lit ; si personne n'ecrit la
convention, aucun detecteur ne remplacera son absence. L'affordance
est la seule des trois classes ou la sonde depend d'une decision
humaine amont — la convention — et c'est une dependance a documenter
dans le rapport, pas a taire.

La page `affordance` echoue : ses deux CTA n'ont pas de classe de bouton, et
leurs opacites (0,5 et 0,4) sont sous le seuil. Un visiteur ne comprend pas que
ce sont des actions. La encore, le smoke test avait seulement verifie qu'un
`<a>` existait — il n'a aucune idee de son rendu.

## 7. La matrice — ce que chaque sonde voit

On croise maintenant les quatre detecteurs sur les quatre pages. C'est le
resultat central du notebook.

In [8]:
def conformite_globale(page):
    sm = smoke_test(page)
    smoke_ok = (sm["structure"] == "PASS" and sm["action"] == "PASS")
    _, v_contraste = audit_contraste_texte(page, CHARTE["fond"])
    v_dominance = "PASS" if not audit_dominance(page) else "FAIL"
    aff = audit_affordance(page)
    v_affordance = "PASS" if (aff and all(x[3] == "PASS" for x in aff)) else "FAIL"
    return {
        "smoke":      "PASS" if smoke_ok else "FAIL",
        "contraste":  v_contraste,
        "dominance":  v_dominance,
        "affordance": v_affordance,
    }

print("page".ljust(11), "smoke", "contraste", "dominance", "affordance", "  GLOBAL", sep="   ")
print("-" * 70)
for nom, page in PAGES.items():
    v = conformite_globale(page)
    global_v = "PASS" if all(x == "PASS" for x in v.values()) else "FAIL"
    print(nom.ljust(11), v["smoke"].ljust(6), v["contraste"].ljust(9),
          v["dominance"].ljust(10), v["affordance"].ljust(10), " ", global_v)

page          smoke   contraste   dominance   affordance     GLOBAL
----------------------------------------------------------------------
conforme    PASS   PASS      PASS       PASS         PASS
primaires   PASS   PASS      FAIL       PASS         FAIL
contraste   PASS   FAIL      PASS       PASS         FAIL
affordance  PASS   PASS      PASS       FAIL         FAIL


### La matrice se lit par colonnes, puis par lignes

La sortie merite les deux lectures, dans cet ordre.

**Par colonnes d'abord.** `smoke` : quatre PASS — une colonne
constante ne discrimine aucune page, c'est la definition d'une sonde
aveugle sur ce banc, et la sortie en porte la preuve formelle.
`contraste` : une seule rouge (`contraste`). `dominance` : une seule
rouge (`primaires`). `affordance` : une seule rouge (`affordance`).
Chaque detecteur discrimine exactement la page portant sa classe de
defaut — ni plus, ni moins. **Par lignes ensuite.** Chaque page
defaillante porte exactement une case rouge, jamais deux : les trois
detecteurs ne se recouvrent sur aucune page. Croisees, les deux
lectures prouvent deux choses differentes : la lecture par colonnes
prouve que chaque detecteur **voit** sa classe ; la lecture par lignes
prouve que les detecteurs sont **orthogonaux** — aucun n'est redundant
avec un autre, donc aucun ne peut etre supprime sans creer un trou.
C'est la difference entre un banc et une collection de gadgets : chaque
sonde ajoute une colonne que personne d'autre ne fournit.

Le GLOBAL, lui, est un ET simple : une seule case rouge coule la page.
Refuser ici la tentation de l'agregat flatteur : la page `primaires`
est verte sur trois des quatre colonnes — un tel pourcentage serait un
chiffre exact et un verdict faux, parce qu'un defaut de palette n'est
pas rachetable par un bon contraste. Un seul ET, pas une moyenne : la
conformite visuelle n'est pas compensatoire. Et la limite de la
matrice, deja preparee a la section 4 : elle ne certifie que les
classes de defaut qu'elle couvre — un defaut de contraste cache dans
un badge hors regex passerait les quatre colonnes. La matrice est
complete contre ses trois classes, pas contre tout.
La matrice est aussi le plan d'extension de l'audit. Ajouter une
capacite — controler la police, les espacements, la comportement
responsive — se fait par **colonnes** : une classe de defaut de plus,
un detecteur de plus, et une page de test de plus portant exactement
ce defaut. La fixture grandit avec la matrice, ligne par ligne, et
c'est ce qui tient le cout lineaire : grace a l'orthogonalite
demontree par les lignes, aucune nouvelle colonne n'exige de retester
les autres classes — chaque classe a sa page, chaque page sa colonne
rouge attendue. Une matrice non orthogonale (deux detecteurs
attrapant le meme defaut) couterait le double pour la meme
information ; la lecture par lignes du banc est donc aussi un test de
**rendement** : chaque colonne ajoutee doit discriminer une page
qu'aucune colonne existante ne discriminait. Sinon, elle ne fait pas
partie de la matrice — elle fait doublon quelque part.

**Lecture.** La colonne `smoke` est verte partout — y compris sur les trois
pages cassees. Chaque page defaillante est rathee par **exactement un** des
trois detecteurs visuels, et conforme pour tous les autres. La colonne
`GLOBAL` (conformite visuelle) est la seule qui discrimine, et elle exige la
reunion des quatre sondes.

C'est la lecon, mesuree :

> **Un smoke test structurel ne peut pas, par construction, detecter un
> defaut de conformite visuelle.** Il mesure le contenant (la structure est
> presente), pas le contenu (la structure respecte-t-elle la charte ?).
> Declaring une page « operationnelle » sur la foi du seul smoke test, c'est
> declarer sain ce qu'on n'a pas regarde.

Et comme l'ecrit le document de reference cite en introduction : le motif
recidive. La classe *systeme* (HTTP 200 sur un site casse) et la classe
*visuelle* (smoke structurel vert sur une UI cassee) sont deux instances du
meme defaut de methode : **confondre « la sonde a passe » avec « l'etat est
vrai ».**

## 8. Pourquoi l'agent ne peut pas s'auto-certifier

Un Copilot Gutenberg qui regenere un template peut-il, lui-meme, certifier que
le rendu est conforme ? Non, pour deux raisons que la matrice rend tangibles.

1. **L'agent optimise ce qu'on mesure.** Si l'auto-certification porte sur le
   smoke test (structure presente), l'agent produira des pages qui passent le
   smoke test — bleu Bootstrap compris. Le detecteur devient un objectif, et
   l'objectif est trop pauvre pour garantir la conformite.

2. **La conformite visuelle est un jugement externe.** Elle suppose une charte
   declaree (section 1) et des detecteurs dedies (sections 4 a 6) qu'un agent
   generique n'embarque pas. La charte est un contrat entre le lieu d'edition
   et le lieu de rendu ; l'agent operate au deuxieme sans connaitre le premier.

La consequence pratique : apres toute modification de template par un agent,
**un humain (ou un harnais d'audit dedie) doit faire tourner les detecteurs
visuels** avant de declarer le rendu operationnel. Confier cette etape a
l'agent lui-meme, c'est lui demander de noter sa propre copie.

### Ce que ces detecteurs ne mesurent pas

L'honnetete du banc se joue ici : enumerer ce qui reste hors champ
apres les quatre colonnes. Trois frontieres majeures.

**Le rendu, d'abord.** Tous les detecteurs lisent le HTML **source**
par expressions regulieres — jamais le rendu calcule. La cascade CSS,
l'heritage des proprietes, les regles d'une feuille de style externe,
un `display:none` qui masque un CTA que l'audit notera PASS : rien de
tout cela n'existe pour une regex. La ou le notebook lit un attribut
`style` inline, un audit reel interrogerait le navigateur — rendu
calcule, propriete effective, capture d'ecran. La difference n'est pas
cosmetique : deux pages au HTML identique rendues sous deux feuilles
de style differentes recoivent ici le meme verdict.

**La typographie, ensuite.** La charte declare `Playfair Display` pour
les titres ; aucun detecteur ne verifie la police reellement employee.
Une page dont les titres rendent en sans-serif generique — defaut
d'identite visible au premier regard — passe les quatre colonnes. La
police est une cellule de la charte sans detecteur associe : a ajouter
a une version de production (par exemple en comparant les familles
declarees du rendu aux familles de la charte).

**La multiplicite des fonds, enfin.** L'audit de contraste suppose une
couleur de fond unique, passee en argument. Texte sur tuile coloree,
sur image, sur degrade : hors perimetre. Le banc le sait — c'est
pourquoi les pages de test posent tout leur texte sur le creme de
fond — mais une page reelle ne s'y pliera pas d'elle-meme.

La conclusion n'est pas « ces detecteurs sont incomplets, abandonnons
au jugement humain » : c'est « chaque classe de defaut exige sa
sonde, et toute sonde a un perimetre fini — donc un audit est une
**collection de sondes nommees**, jamais un outil unique ». Le
jugement humain garde ce que la machine n'a pas encore appris a
nommer ; il ne devrait plus garder ce qu'elle sait enumerer.
Le chemin de maturation est balise, et chaque etape ne change pas la
structure de la matrice — seulement l'implementation des colonnes.
Etape 1 : remplacer les regex par un **navigateur headless** —
`getComputedStyle` donne la propriete effective, cascade et heritage
resolus ; les colonnes restent, leurs perimetres se retractent. Etape
2 : ajouter la **comparaison au rendu** — captures d'ecran couplees a
la charte rendue, pour les classes (typographie, composition) que le
DOM seul ne porte pas. Etape 3 : la **derive temporelle** — rejouer
la matrice a chaque deploiement, parce qu'une conformite est un etat,
pas une propriete : chaque reconfiguration peut defaire un cablage
correct, et la matrice ne sert qu'autant qu'elle est reexecutee.
Le schema de ce notebook est volontairement en etape 0 : ce qu'il
enseigne — une question par sonde, un perimetre explicite, un
verdict non compensatoire — est ce qui rendra les etapes 1 a 3
exploitables, faute de quoi elles produiront des scores opaques.

## 9. Exercices

Les trois exercices suivants sont laisses a completer. Convention : un stub
retourne `None` ou `pass` — il ne levere pas d'exception, et s'execute sans
erreur.

### Le programme des trois exercices

Les trois exercices ne sont pas trois taches du meme genre : chacun
exerce une competence differente de la chaine d'audit, et leur ordre
est une progression.

L'exercice 1 demande les primaires **avec leur contexte** — pas
seulement le compte que la section 5 affiche deja. Enumerer ou vit
chaque couleur importee (CTA, badge, bordure ?) fait passer le
detecteur du temoin a la **gravite** : un bleu en CTA principal et le
meme bleu en filet decoratif sont le meme temoin d'import mais pas la
meme urgence. C'est la competence de description — dire precisement ce
qui est casse avant de vouloir le reparer.

L'exercice 2 fait construire le rapport agrege — et c'est le piege le
plus formateur des trois : la tentation naturelle est un pourcentage
de conformite (trois colonnes vertes sur quatre), que la matrice de la
section 7 a explicitement disqualifie. Construire un agregat qui ne
mente pas — pire cas par classe de detection, jamais moyenne entre
classes — oblige a decider ce que « conforme » veut dire operationnellement. La competence visee est l'agregation honnete : la meme
mesure, plusieurs facons de la resumer, dont certaines mentent.

L'exercice 3 demande la page qui passe tout — le smoke y compris —
tout en etant visuellement cassee. Pour y arriver, il faut avoir
compris les perimetres des sections 4, 5 et 6 assez finement pour
placer un defaut dans l'angle mort de chaque regex (un contraste
invalide hors `p`/`h*`, une couleur hors denylist, un libelle hors
liste). Ecrire le casse qui passe est la competence la plus haute de
l'audit : celui qui sait pieger la sonde est le seul a savoir ou elle
ne protege pas. Mesurer, agreger, pieger — dans cet ordre.
Mode d'emploi des squelettes : chaque exercice est une cellule code
avec un commentaire d'enonce et un retour `None` — la competence
visee est nommee, l'indice fourni, la correction deliberement
absente. C'est un choix, pas une economie : la preuve d'aptitude en
audit n'est pas de lire une correction mais de produire un verdict
defendable — et le verdict se defend d'abord contre son auteur. Le
critere de reussite de l'exercice 3 merite d'etre donne d'avance,
parce qu'il est contre-intuitif : la page piege **reussit** si elle
passe les quatre colonnes de la matrice tout en etant visuellement
cassee. Celui qui l'ecrit vient de construire, de ses mains, un
angle mort — et l'avoir construit est la seule facon fiable de s'en
souvenir quand on en rencontrera un sans l'avoir construit.

In [9]:
# Exercice 1 — Enumerer les primaires avec leur contexte.
# Ecrire detecteur_primaire_contexte(html_str) qui retourne une liste de
# tuples (element, couleur) indiquant dans quelle balise chaque primaire
# Bootstrap apparait. Retourner [] si aucune.
def detecteur_primaire_contexte(html_str):
    # Parcourir les balises, extraire leur couleur de fond ou de texte,
    # croiser avec PRIMAIRES_BOOTSTRAP, retourner le contexte.
    return []

In [10]:
# Exercice 2 — Rapport agrege.
# Ecrire rapport_conformite(html_str, charte) qui retourne un dictionnaire
# {contraste, dominance, affordance} donnant le verdict et un resume humain
# pour chacun. Retourner un dict vide si la page est vide.
def rapport_conformite(html_str, charte):
    # Reunir les trois detecteurs en un rapport lisible.
    return {}

In [11]:
# Exercice 3 — Le piege du smoke vert.
# Etant donne une page qui PASS le smoke_test, ecrire piege_smoke_vert(page)
# qui retourne la classe du defaut visuel parmi {"contraste", "dominance",
# "affordance", None}. Retourner None si la page est conforme.
def piege_smoke_vert(page):
    # On suppose deja que smoke_test(page) passe ; chercher le defaut.
    return None

## Ce qu'il faut retenir

- **Le smoke test structurel mesure le contenant, pas la conformite.** Une page
  au rendu casse le passe tranquillement.
- **Chaque classe de defaut visuel demande son propre detecteur** : contraste
  WCAG (luminance, pas teinte), dominance des primaires (croisement avec la
  charte), affordance des CTA (classe + opacite). Aucun ne suffit seul.
- **La conformite visuelle est un jugement externe** : elle suppose une charte
  declaree et ne peut pas etre auto-certifiee par l'agent qui genere le rendu.
- Ce motif est la classe *visuelle* du « probe menteur » decrit dans
  `docs/reference/verification-verte-systeme-casse.md` (classe *systeme*).
  La structure recidive : une sonde qui passe ne prouve pas que l'etat est vrai.

*Fixture synthetique « Maison Valmont » — couleurs, polices et textes fictifs.
Aucune donnee client, aucun nom reel, aucun secret. Toutes les fonctions sont
deterministes (stdlib seul : `re`, pas de cle, pas de reseau).*

### D'ou vient cette matrice

Une derniere lecture, en guise de provenance. La structure de ce
notebook — une sonde usuelle aveugle, puis une sonde par classe de
defaut — n'est pas nee d'une speculation : elle condense un incident
documente dans le depot, ou des audits consecutifs d'un site reel ont
declare « operationnel » en se fondant sur une sonde structurelle,
chaque episode corrigeant la sonde precedente sans jamais corriger la
question posee. L'etude de cas complete vit dans
`docs/reference/verification-verte-systeme-casse.md` — six classes de
sondes menteuses y sont cataloguees avec leur signature et leur
contre-mesure ; la classe visuelle y est la soeur de la classe
systeme (un statut 200 sur un site casse).

Ce notebook en est la mise en forme executable pour la classe
visuelle : quatre pages, trois defauts, une sonde aveugle par
conception, trois detecteurs orthogonaux, une matrice qui refuse
l'agregat flatteur. La transposition a un projet reel suit
exactement les sections : ecrire la charte en donnees (section 1),
denombrer les classes de defaut observes (section 2), nommer la
question exacte a laquelle chaque sonde existante repond (section 3),
puis une sonde par classe, chacune avec son perimetre explicite
(sections 4 a 6), et un verdict non compensatoire (section 7).

Et la regle la plus portable reste celle de la section 3 : avant
d'invoquer une sonde comme preuve, nommer la question a laquelle elle
repond. Le smoke test n'a jamais menti dans cette histoire — il a
toujours repondu exactement, sincerement, a une question que personne
n'avait formulee. Le defaut etait dans la citation, pas dans la
mesure.
Une precision de citation, pour finir. Ce notebook est un **schema de
preuve**, pas un precedent : une equipe qui l'applique a son projet
ne doit pas ecrire « conforme au notebook » mais nommer ses colonnes
(« contraste : pire cas 9.02 ; dominance : zero primaire ; affordance :
2 CTA verifies »), ses perimetres et ses N/A — la matrice sert a
rendre le rapport **falsifiable**, pas a le raccourcir. Quant a
l'ordre des sections, il n'est pas pedagogique seulement : il suit
l'ordre historique des decouvertes de l'incident origine — la sonde
usuelle d'abord, puis chaque classe de defaut le jour ou elle a ete
vue, puis la matrice le jour ou l'on a compris qu'aucune sonde
isolee ne suffirait. Un audit qui commence par la matrice gagne du
temps ; un audit qui commence par l'incident comprend pourquoi la
matrice a cette forme — et saura en changer la forme le jour ou une
septieme classe de defaut apparaitra.